# MOE8 LOGO-Lite — gate-only target-selection experiment
Attach the **XACLE_2027_data** and **moescore_full8_artifacts** Kaggle datasets. This notebook checks out the experiment branch, verifies the Full8 cache, runs one operational smoke fold, then trains all eligible generator folds. No expert model is loaded or extracted. Outputs are under `/kaggle/working/moescore_full8_logo_lite_artifacts/`.

LOGO-Lite is a research proxy, not strict domain generalization. Frozen experts may have seen held-out generators, and held-out train labels select checkpoints.


In [ ]:
from pathlib import Path
import csv, json, os, shutil, subprocess, sys

BRANCH = "experiment/moe8-logo-lite"
SOURCE_COMMIT = None  # Optional exact implementation commit SHA.
DATASET_NAME = "XACLE_2027_data"
ARTIFACTS_NAME = "moescore_full8_artifacts"
DATASET_ROOT_OVERRIDE = None
ARTIFACTS_ROOT_OVERRIDE = None
RESUME_ARTIFACTS_OVERRIDE = None  # Previously exported LOGO output directory, if attached.
RUN_MODE = "all"  # all, single, selected
GENERATOR_IDS = []
SEED = 2027
DEVICE = "cuda:0"
RESUME = True
DRY_RUN = False
OUTPUT_DIR = Path("/kaggle/working/moescore_full8_logo_lite_artifacts")
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
REPO_ROOT = Path("/kaggle/working/MOESCORE_LOGO")

assert RUN_MODE in {"all", "single", "selected"}
if RUN_MODE == "single": assert len(GENERATOR_IDS) == 1
if RUN_MODE == "selected": assert GENERATOR_IDS


In [ ]:
def run(command, **kwargs):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), check=True, **kwargs)

for module in ("torch", "numpy", "scipy", "yaml", "matplotlib"):
    __import__(module)
if not REPO_ROOT.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_ROOT])
else:
    actual = subprocess.check_output(["git", "-C", str(REPO_ROOT), "branch", "--show-current"], text=True).strip()
    assert actual == BRANCH, f"Existing checkout is on {actual}, expected {BRANCH}"
    run(["git", "-C", REPO_ROOT, "pull", "--ff-only"])
if SOURCE_COMMIT:
    run(["git", "-C", REPO_ROOT, "checkout", SOURCE_COMMIT])
actual_commit = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Implementation commit:", actual_commit)
sys.path.insert(0, str(REPO_ROOT))

def attached(name, marker, override):
    if override:
        candidate = Path(override)
        assert candidate.is_dir(), candidate
        return candidate
    normalized = lambda s: "".join(c for c in s.lower() if c.isalnum())
    candidates = set()
    for marker_path in Path("/kaggle/input").rglob(marker):
        for parent in marker_path.parents:
            if normalized(parent.name) == normalized(name):
                candidates.add(parent)
                break
    assert len(candidates) == 1, f"Expected one attached {name} containing {marker}; found {sorted(candidates)}"
    return candidates.pop()

DATASET_ROOT = attached(DATASET_NAME, "train_average.csv", DATASET_ROOT_OVERRIDE)
ARTIFACTS_ROOT = attached(ARTIFACTS_NAME, "moe_full8.yaml", ARTIFACTS_ROOT_OVERRIDE)
print("Dataset:", DATASET_ROOT)
print("Full8 artifacts:", ARTIFACTS_ROOT)
if RESUME_ARTIFACTS_OVERRIDE:
    assert RESUME and not OUTPUT_DIR.exists(), "Restore requires RESUME=True and an empty output destination"
    shutil.copytree(RESUME_ARTIFACTS_OVERRIDE, OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
script = REPO_ROOT / "scripts/train_moe8_logo.py"
config = REPO_ROOT / "configs/moe_full8_logo.example.yaml"
base = [sys.executable, script, "--config", config, "--dataset-root", DATASET_ROOT,
        "--artifacts-root", ARTIFACTS_ROOT, "--output-dir", OUTPUT_DIR, "--device", DEVICE, "--seed", SEED]
run(base + ["--dry-run"])
with (OUTPUT_DIR / "generator_statistics.csv").open() as stream:
    eligible = [row["model_id"] for row in csv.DictReader(stream) if row["role"] == "eligible_generator"]
assert eligible, "No eligible generator folds"
print("Eligible generator IDs:", eligible)
if not DRY_RUN:
    selected = eligible if RUN_MODE == "all" else GENERATOR_IDS
    assert all(generator_id in eligible for generator_id in selected)
    smoke = base + ["--generator-id", selected[0], "--smoke-epochs", "1"]
    if RESUME: smoke += ["--resume"]
    run(smoke)
    full = base + (["--all-generators"] if RUN_MODE == "all" else
                   [argument for generator_id in selected for argument in ("--generator-id", generator_id)])
    if RESUME: full += ["--resume"]
    run(full)
    print((OUTPUT_DIR / "summary" / "report.md").read_text())
